# GASNet Pipeline on VRU (PyTorch)

Notebook này triển khai đầy đủ pipeline theo yêu cầu:

1. Đọc và tổ chức dữ liệu VRU theo các split chuẩn.
2. Tạo query/gallery cho từng test set (small/medium/large).
3. Xây dựng mô hình GASNet (ResNet50 + Global Attention + Full-Scale branch + BNNeck).
4. Huấn luyện với Cross-Entropy + Triplet Loss, optimizer Adam.
5. Đánh giá bằng mAP, Rank-1, Rank-5.

Lưu ý: Huấn luyện 60 epochs với batch size lớn có thể rất lâu. Hãy chạy theo từng bước và điều chỉnh batch size theo GPU.

In [1]:
import os
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Speed optimizations for fixed-size inputs on modern GPUs (A100/RTX)
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision("high")

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
bf16_supported = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
print(f"GPU: {gpu_name}")
print(f"bf16_supported={bf16_supported}")

# Colab: mount Google Drive if available
try:
    from google.colab import drive  # type: ignore
    if not Path("/content/drive").exists():
        drive.mount("/content/drive")
except Exception:
    pass

# Your dataset location on Colab
ROOT = Path("/content/drive/MyDrive")
VRU_DIR = ROOT / "VRU"
PIC_DIR = VRU_DIR / "Pic"
SPLIT_DIR = VRU_DIR / "train_test_split"

assert PIC_DIR.exists(), f"Missing image folder: {PIC_DIR}"
assert SPLIT_DIR.exists(), f"Missing split folder: {SPLIT_DIR}"
print(f"Paths are valid. VRU_DIR={VRU_DIR}")
if torch.cuda.is_available():
    print("cuDNN benchmark=ON, TF32=ON")

Mounted at /content/drive
Paths are valid. VRU_DIR=/content/drive/MyDrive/VRU


In [2]:
@dataclass
class Sample:
    img_path: Path
    vehicle_id: int
    image_id: int


def preview_split(file_path: Path, n: int = 5) -> None:
    print(f"Preview {file_path.name}:")
    with file_path.open("r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= n:
                break
            print(line.strip())
    print("-")


def read_split_file(file_path: Path) -> List[Sample]:
    samples: List[Sample] = []
    with file_path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            img_id_str, veh_id_str = line.split()
            image_id = int(img_id_str)
            vehicle_id = int(veh_id_str)
            img_path = PIC_DIR / f"{image_id}.jpg"
            samples.append(Sample(img_path=img_path, vehicle_id=vehicle_id, image_id=image_id))
    return samples


preview_split(SPLIT_DIR / "train_list.txt")
preview_split(SPLIT_DIR / "test_list_1200.txt")


train_samples = read_split_file(SPLIT_DIR / "train_list.txt")
test_small_samples = read_split_file(SPLIT_DIR / "test_list_1200.txt")
test_medium_samples = read_split_file(SPLIT_DIR / "test_list_2400.txt")
test_large_samples = read_split_file(SPLIT_DIR / "test_list_8000.txt")


print("Head of train samples:")
for s in train_samples[:5]:
    print(s.image_id, s.vehicle_id, s.img_path.name)

Preview train_list.txt:
152019 12940
152014 12940
152017 12940
152028 12940
152030 12940
-
Preview test_list_1200.txt:
151247 12888
151243 12888
151252 12888
151251 12888
151249 12888
-
Head of train samples:
152019 12940 152019.jpg
152014 12940 152014.jpg
152017 12940 152017.jpg
152028 12940 152028.jpg
152030 12940 152030.jpg


In [3]:
def summarize_samples(samples: List[Sample], name: str, check_missing: bool = False) -> Dict[str, int]:
    img_count = len(samples)
    vid_count = len({s.vehicle_id for s in samples})

    missing_count = -1
    io_error_count = 0

    if check_missing:
        missing = 0
        io_err = 0
        for s in samples:
            try:
                if not s.img_path.exists():
                    missing += 1
            except OSError:
                io_err += 1
                missing += 1
        missing_count = missing
        io_error_count = io_err

    stats = {
        "images": img_count,
        "vehicles": vid_count,
        "missing_images": missing_count,
        "io_errors": io_error_count,
    }

    if check_missing:
        print(f"[{name}] images={img_count}, vehicles={vid_count}, missing_images={missing_count}, io_errors={io_error_count}")
    else:
        print(f"[{name}] images={img_count}, vehicles={vid_count}, missing_images=SKIPPED")

    return stats


expected = {
    "train": (80532, 7085),
    "small": (13920, 1200),
    "medium": (27345, 2400),
    "large": (91595, 8000),
}

# Colab + Google Drive de tranh loi I/O khi check ton tai tat ca file, mac dinh bo qua check_missing
CHECK_MISSING = True

stats_train = summarize_samples(train_samples, "train", check_missing=CHECK_MISSING)
stats_small = summarize_samples(test_small_samples, "test_small", check_missing=CHECK_MISSING)
stats_medium = summarize_samples(test_medium_samples, "test_medium", check_missing=CHECK_MISSING)
stats_large = summarize_samples(test_large_samples, "test_large", check_missing=CHECK_MISSING)

print("\nExpected-vs-actual checks:")
for key, (exp_img, exp_vid) in expected.items():
    actual = {
        "train": stats_train,
        "small": stats_small,
        "medium": stats_medium,
        "large": stats_large,
    }[key]
    ok_img = actual["images"] == exp_img
    ok_vid = actual["vehicles"] == exp_vid
    print(f"{key}: images {'OK' if ok_img else 'MISMATCH'} ({actual['images']} vs {exp_img}), "
          f"vehicles {'OK' if ok_vid else 'MISMATCH'} ({actual['vehicles']} vs {exp_vid})")

[train] images=80533, vehicles=7086, missing_images=6, io_errors=6
[test_small] images=13920, vehicles=1200, missing_images=0, io_errors=0
[test_medium] images=27345, vehicles=2400, missing_images=0, io_errors=0
[test_large] images=91595, vehicles=8000, missing_images=0, io_errors=0

Expected-vs-actual checks:
train: images MISMATCH (80533 vs 80532), vehicles MISMATCH (7086 vs 7085)
small: images OK (13920 vs 13920), vehicles OK (1200 vs 1200)
medium: images OK (27345 vs 27345), vehicles OK (2400 vs 2400)
large: images OK (91595 vs 91595), vehicles OK (8000 vs 8000)


In [4]:
def build_query_gallery(samples: List[Sample]) -> Tuple[List[Sample], List[Sample]]:
    by_vehicle: Dict[int, List[Sample]] = {}
    for s in samples:
        by_vehicle.setdefault(s.vehicle_id, []).append(s)

    query: List[Sample] = []
    gallery: List[Sample] = []

    for vehicle_id, group in by_vehicle.items():
        group_sorted = sorted(group, key=lambda x: x.image_id)
        gallery.append(group_sorted[0])
        query.extend(group_sorted[1:])

    return query, gallery


q_small, g_small = build_query_gallery(test_small_samples)
q_medium, g_medium = build_query_gallery(test_medium_samples)
q_large, g_large = build_query_gallery(test_large_samples)

print(f"Small  -> query: {len(q_small)}, gallery: {len(g_small)}")
print(f"Medium -> query: {len(q_medium)}, gallery: {len(g_medium)}")
print(f"Large  -> query: {len(q_large)}, gallery: {len(g_large)}")

Small  -> query: 12720, gallery: 1200
Medium -> query: 24945, gallery: 2400
Large  -> query: 83595, gallery: 8000


In [5]:
class VRUDataset(Dataset):
    def __init__(self, samples: List[Sample], transform=None, relabel=False, label_map=None, max_retries: int = 3):
        self.samples = samples
        self.transform = transform
        self.relabel = relabel
        self.label_map = label_map or {}
        self.max_retries = max_retries

    def __len__(self):
        return len(self.samples)

    def _load_rgb(self, path: Path):
        last_err = None
        for _ in range(self.max_retries):
            try:
                with Image.open(path) as img:
                    return img.convert("RGB")
            except OSError as e:
                last_err = e
        raise last_err

    def __getitem__(self, idx):
        # Google Drive tren Colab co the bi I/O timeout theo tung file, fallback qua mau khac de khong lam vo DataLoader
        tried = 0
        cur_idx = idx
        while tried < min(10, len(self.samples)):
            s = self.samples[cur_idx]
            try:
                img = self._load_rgb(s.img_path)
                if self.transform is not None:
                    img = self.transform(img)
                label = self.label_map[s.vehicle_id] if self.relabel else s.vehicle_id
                return img, label, s.vehicle_id, s.image_id
            except OSError:
                tried += 1
                cur_idx = (cur_idx + 1) % len(self.samples)

        raise RuntimeError(f"Cannot read image after retries, start_idx={idx}, path={self.samples[idx].img_path}")


train_vehicle_ids = sorted({s.vehicle_id for s in train_samples})
train_label_map = {vid: i for i, vid in enumerate(train_vehicle_ids)}
num_classes = len(train_label_map)
print(f"Num train classes: {num_classes}")

Num train classes: 7086


In [6]:
train_tfms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomCrop((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

test_tfms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def make_loader(samples: List[Sample], batch_size: int, is_train: bool, relabel=False):
    ds = VRUDataset(
        samples=samples,
        transform=train_tfms if is_train else test_tfms,
        relabel=relabel,
        label_map=train_label_map if relabel else None,
        max_retries=3,
    )

    is_colab_drive = str(PIC_DIR).startswith("/content/drive")
    workers = 0 if is_colab_drive else 2

    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=is_train,
        num_workers=workers,
        pin_memory=torch.cuda.is_available(),
        drop_last=is_train,
    )


BASE_BATCH_SIZE = 128
BASE_LR = 0.00035

# A100 can often handle much larger batch sizes
if torch.cuda.is_available():
    TRAIN_BATCH_SIZE = 512
else:
    TRAIN_BATCH_SIZE = 64

EVAL_BATCH_SIZE = 512
LEARNING_RATE = BASE_LR * (TRAIN_BATCH_SIZE / BASE_BATCH_SIZE)

train_loader = make_loader(train_samples, TRAIN_BATCH_SIZE, is_train=True, relabel=True)
q_small_loader = make_loader(q_small, EVAL_BATCH_SIZE, is_train=False)
g_small_loader = make_loader(g_small, EVAL_BATCH_SIZE, is_train=False)
q_medium_loader = make_loader(q_medium, EVAL_BATCH_SIZE, is_train=False)
g_medium_loader = make_loader(g_medium, EVAL_BATCH_SIZE, is_train=False)
q_large_loader = make_loader(q_large, EVAL_BATCH_SIZE, is_train=False)
g_large_loader = make_loader(g_large, EVAL_BATCH_SIZE, is_train=False)

print("DataLoaders are ready.")
print(f"num_workers={'0 (Colab Drive safe mode)' if str(PIC_DIR).startswith('/content/drive') else '2'}")
print(f"TRAIN_BATCH_SIZE={TRAIN_BATCH_SIZE}, EVAL_BATCH_SIZE={EVAL_BATCH_SIZE}, LR={LEARNING_RATE:.6f}")

DataLoaders are ready.


## GASNet Model: ResNet50 + GA + Full-Scale + BNNeck

In [ ]:
import warnings

# Harmless torch.compile(Inductor) warning: suppress repeated "online softmax" messages
warnings.filterwarnings(
    "ignore",
    message=r".*Online softmax is disabled on the fly.*",
    category=UserWarning,
    module=r"torch\._inductor\.lowering",
)

print("Inductor softmax warning is suppressed.")

In [7]:
class RGABlock(nn.Module):
    def __init__(self, channels: int, reduction: int = 16, spatial_size: int = 7):
        super().__init__()
        mid = max(1, channels // reduction)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.c_mlp = nn.Sequential(
            nn.Conv2d(channels, mid, kernel_size=1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid, channels, kernel_size=1, bias=False),
        )
        self.s_mlp = nn.Sequential(
            nn.Conv2d(1, mid, kernel_size=1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid, 1, kernel_size=1, bias=False),
        )
        self.spatial_size = spatial_size
        self.sigmoid = nn.Sigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        c = self.c_mlp(self.pool(x))
        c = self.sigmoid(c)
        s = x.mean(dim=1, keepdim=True)
        s = F.adaptive_avg_pool2d(s, self.spatial_size)
        s = self.s_mlp(s)
        s = F.interpolate(s, size=x.shape[-2:], mode="bilinear", align_corners=False)
        s = self.sigmoid(s)
        return x * c * s


class AggregationGate(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
            ),
            nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
            ),
            nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=5, padding=2, bias=False),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
            ),
            nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=2, dilation=2, bias=False),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
            ),
        ])
        self.gate = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(out_ch * 4, out_ch, kernel_size=1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, 4, kernel_size=1, bias=True),
            nn.Softmax(dim=1),
        )
        self.fuse = nn.Sequential(
            nn.Conv2d(out_ch, out_ch, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feats = [b(x) for b in self.branches]
        stacked = torch.cat(feats, dim=1)
        weights = self.gate(stacked)
        w1, w2, w3, w4 = [weights[:, i:i+1] for i in range(4)]
        out = feats[0] * w1 + feats[1] * w2 + feats[2] * w3 + feats[3] * w4
        return self.fuse(out)


class FullScaleBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.agg = AggregationGate(in_ch, out_ch)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.agg(x)


class GASNet(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        base = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        self.stem = nn.Sequential(base.conv1, base.bn1, base.relu, base.maxpool)
        self.layer1 = base.layer1
        self.layer2 = base.layer2
        self.layer3 = base.layer3
        self.layer4 = base.layer4
        self.ga_blocks = nn.ModuleList([RGABlock(512) for _ in range(len(self.layer2))])
        self.fs = FullScaleBlock(1024, 512)
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.bnneck_global = nn.BatchNorm1d(2048)
        self.bnneck_fs = nn.BatchNorm1d(512)
        self.classifier_global = nn.Linear(2048, num_classes, bias=False)
        self.classifier_fs = nn.Linear(512, num_classes, bias=False)

    def forward(self, x: torch.Tensor):
        x = self.stem(x)
        x = self.layer1(x)
        for block, ga in zip(self.layer2, self.ga_blocks):
            x = block(x)
            x = ga(x)
        x = self.layer3(x)
        fs = self.fs(x)
        x = self.layer4(x)
        global_feat = self.gap(x).flatten(1)
        fs_feat = self.gap(fs).flatten(1)
        bn_global = self.bnneck_global(global_feat)
        bn_fs = self.bnneck_fs(fs_feat)
        if self.training:
            logits_g = self.classifier_global(bn_global)
            logits_f = self.classifier_fs(bn_fs)
            return (global_feat, fs_feat), (logits_g, logits_f)
        return (global_feat, fs_feat), (bn_global, bn_fs)


def batch_hard_triplet_loss(feat: torch.Tensor, labels: torch.Tensor, margin: float = 0.3) -> torch.Tensor:
    dist = torch.cdist(feat, feat, p=2)
    labels = labels.view(-1, 1)
    mask_pos = labels.eq(labels.t())
    mask_neg = ~mask_pos
    dist_pos = dist.clone()
    dist_pos[~mask_pos] = -1.0
    dist_pos.fill_diagonal_(-1.0)
    hardest_pos, _ = dist_pos.max(dim=1)
    dist_neg = dist.clone()
    dist_neg[~mask_neg] = 1e9
    hardest_neg, _ = dist_neg.min(dim=1)
    valid = hardest_pos > -0.5
    if valid.sum() == 0:
        return torch.tensor(0.0, device=feat.device)
    loss = F.relu(hardest_pos[valid] - hardest_neg[valid] + margin)
    return loss.mean()


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    epoch: int,
    scaler,
    use_amp: bool = True,
    amp_dtype: torch.dtype = torch.bfloat16,
    use_channels_last: bool = True,
    log_every: int = 20,
):
    import time

    model.train()
    ce_loss = nn.CrossEntropyLoss()
    running = 0.0
    n_steps = len(loader)
    t0 = time.time()

    print(f"[Epoch {epoch}] start | steps={n_steps} | batch={loader.batch_size} | amp={use_amp} | amp_dtype={amp_dtype}")

    for step, (imgs, labels, _, _) in enumerate(loader, 1):
        t_step = time.time()
        if use_channels_last:
            imgs = imgs.to(device, non_blocking=True, memory_format=torch.channels_last)
        else:
            imgs = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.autocast(device_type="cuda", dtype=amp_dtype, enabled=use_amp):
            (g_feat, f_feat), (logits_g, logits_f) = model(imgs)
            loss_id = ce_loss(logits_g, labels) + ce_loss(logits_f, labels)
            loss_tri = batch_hard_triplet_loss(g_feat, labels) + batch_hard_triplet_loss(f_feat, labels)
            loss = loss_id + loss_tri

        optimizer.zero_grad(set_to_none=True)
        if use_amp and scaler.is_enabled():
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        running += loss.item() * imgs.size(0)

        if step == 1 or step % log_every == 0 or step == n_steps:
            dt = time.time() - t_step
            gpu_mem = torch.cuda.memory_allocated() / 1024**3 if torch.cuda.is_available() else 0.0
            print(f"  step {step}/{n_steps} | loss={loss.item():.4f} | step_time={dt:.2f}s | gpu_mem={gpu_mem:.2f}GB")

    avg = running / len(loader.dataset)
    print(f"Epoch {epoch}: loss={avg:.4f} | epoch_time={(time.time() - t0)/60:.1f} min")
    return avg


@torch.no_grad()
def extract_features(model: nn.Module, loader: DataLoader, use_bn: bool = True, use_channels_last: bool = True):
    model.eval()
    feats = []
    pids = []
    for imgs, _, vid, _ in loader:
        if use_channels_last:
            imgs = imgs.to(device, memory_format=torch.channels_last)
        else:
            imgs = imgs.to(device)
        (g_feat, f_feat), (bn_g, bn_f) = model(imgs)
        out = torch.cat([bn_g, bn_f], dim=1) if use_bn else torch.cat([g_feat, f_feat], dim=1)
        feats.append(out.cpu())
        pids.append(vid)
    feats = torch.cat(feats, dim=0)
    pids = torch.cat(pids, dim=0).cpu()
    return feats, pids


def evaluate_map_cmc(query_feats: torch.Tensor, query_ids: torch.Tensor, gallery_feats: torch.Tensor, gallery_ids: torch.Tensor, topk=(1, 5)):
    q = F.normalize(query_feats, dim=1)
    g = F.normalize(gallery_feats, dim=1)
    sim = q @ g.t()
    num_q = q.size(0)
    max_k = max(topk)
    cmc = torch.zeros(max_k)
    ap_list = []
    for i in range(num_q):
        order = torch.argsort(sim[i], descending=True)
        matches = (gallery_ids[order] == query_ids[i]).int()
        if matches.sum() == 0:
            continue
        first = torch.where(matches == 1)[0][0].item()
        if first < max_k:
            cmc[first:] += 1
        idxs = torch.where(matches == 1)[0]
        prec = torch.cumsum(matches, 0)[idxs] / (idxs + 1)
        ap_list.append(prec.mean().item())
    cmc = cmc / num_q
    mAP = float(np.mean(ap_list)) if ap_list else 0.0
    r1 = float(cmc[topk[0] - 1])
    r5 = float(cmc[topk[1] - 1]) if max_k >= 5 else 0.0
    return mAP, r1, r5


USE_CHANNELS_LAST = torch.cuda.is_available()
AMP_DTYPE = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16
USE_AMP = torch.cuda.is_available()

model = GASNet(num_classes=num_classes).to(device)
if USE_CHANNELS_LAST:
    model = model.to(memory_format=torch.channels_last)

USE_COMPILE = hasattr(torch, "compile") and torch.cuda.is_available()
if USE_COMPILE:
    try:
        print("Compiling model with torch.compile (first step may be slower)...")
        model = torch.compile(model, mode="reduce-overhead")
    except Exception as e:
        print(f"torch.compile skipped: {e}")

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

# GradScaler helps fp16; for bf16 we keep scaler disabled for stability/simplicity
use_scaler = USE_AMP and AMP_DTYPE == torch.float16
scaler = torch.cuda.amp.GradScaler(enabled=use_scaler)

EPOCHS = 60
RUN_TRAIN = True
RUN_EVAL = False
WEIGHTS_PATH = ROOT / "gasnet_vru.pth"

print(f"USE_AMP={USE_AMP}, AMP_DTYPE={AMP_DTYPE}, USE_CHANNELS_LAST={USE_CHANNELS_LAST}, LR={LEARNING_RATE:.6f}")

if RUN_TRAIN:
    for epoch in range(1, EPOCHS + 1):
        train_one_epoch(
            model,
            train_loader,
            optimizer,
            epoch,
            scaler=scaler,
            use_amp=USE_AMP,
            amp_dtype=AMP_DTYPE,
            use_channels_last=USE_CHANNELS_LAST,
            log_every=20,
        )
    torch.save(model.state_dict(), WEIGHTS_PATH)
    print(f"Saved: {WEIGHTS_PATH}")

if RUN_EVAL:
    if WEIGHTS_PATH.exists():
        model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=device))
    qf, qids = extract_features(model, q_small_loader, use_channels_last=USE_CHANNELS_LAST)
    gf, gids = extract_features(model, g_small_loader, use_channels_last=USE_CHANNELS_LAST)
    mAP, r1, r5 = evaluate_map_cmc(qf, qids, gf, gids)
    print(f"Small  mAP={mAP:.4f} Rank-1={r1:.4f} Rank-5={r5:.4f}")

    qf, qids = extract_features(model, q_medium_loader, use_channels_last=USE_CHANNELS_LAST)
    gf, gids = extract_features(model, g_medium_loader, use_channels_last=USE_CHANNELS_LAST)
    mAP, r1, r5 = evaluate_map_cmc(qf, qids, gf, gids)
    print(f"Medium mAP={mAP:.4f} Rank-1={r1:.4f} Rank-5={r5:.4f}")

    qf, qids = extract_features(model, q_large_loader, use_channels_last=USE_CHANNELS_LAST)
    gf, gids = extract_features(model, g_large_loader, use_channels_last=USE_CHANNELS_LAST)
    mAP, r1, r5 = evaluate_map_cmc(qf, qids, gf, gids)
    print(f"Large  mAP={mAP:.4f} Rank-1={r1:.4f} Rank-5={r5:.4f}")

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 137MB/s] 


KeyboardInterrupt: 